# Transformer inference

Load the latest trained English-to-Italian checkpoint and translate your own sentences. Run `training.ipynb` or `train.py` first so the tokenizer files and at least one checkpoint exist.

In [ ]:
from config import get_config, latest_weights_file_path
from translate import load_inference_bundle, translate

config = get_config()
latest_checkpoint = latest_weights_file_path(config)
print(f"Latest checkpoint: {latest_checkpoint or 'none — train the model first'}")

## Load the model

This rebuilds the Transformer, restores the latest checkpoint, moves it to CUDA/MPS/CPU automatically, and switches it to evaluation mode. To use a particular checkpoint, pass `checkpoint_path=".../tmodel_05.pt"`.

In [ ]:
bundle = load_inference_bundle(config=config, checkpoint_path=None, device="auto")
print(f"Device: {bundle.device}")
print(f"Loaded: {bundle.checkpoint_path.name}")

## Translate your own sentence

Cached decoding begins with `[SOS]` and processes one new token per step without recomputing the decoder prefix. `beam_size=1` is greedy search; a small beam keeps several promising partial translations. Three-gram blocking prevents exact phrase loops.

In [ ]:
sentence = "I am a student."
prediction = translate(
    sentence,
    bundle,
    beam_size=3,
    length_penalty=0.6,
    no_repeat_ngram_size=3,
)

print(f"English: {sentence}")
print(f"Italian: {prediction}")

## Try several examples

The already-loaded `bundle` is reused, so the model is not reconstructed for every sentence. Beam search is slower than greedy search but both paths reuse each decoder layer's KV cache.

In [ ]:
sentences = [
    "Where is the train station?",
    "This book is very interesting.",
    "I would like a cup of coffee.",
]

for text in sentences:
    print(f"{text} -> {translate(text, bundle, beam_size=3, no_repeat_ngram_size=3)}")